# A1.14 · Overwhelming the human in the loop

**Function A — Securing AI Architectures → The Agentic Reference Architecture, and Every Risk It Carries**  ·  *Security of AI*

Builds on **[A1.13 · Repudiation and untraceability](https://spbreed.github.io/cyber-commons/lessons/A1.13.html)**.

| | |
|---|---|
| Open-source tooling | — |
| Open-weight models | — |
| Frontier models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

Approval is a genuine control at four requests a day. At four hundred it is a person clicking approve, and the control has quietly become a log of things somebody scrolled past.

## 2 · The framework

```
   requests/hour     4        40       400
   read carefully   yes      some      no
   approval is    control  friction  a log

   the control does not fail loudly. it degrades into a click.
```

**OWASP T10 — Overwhelming Human-in-the-Loop.**

Human approval is the control everyone reaches for first. It is placed at the
tool call — the right place — and it is genuinely strong for rare, consequential
decisions.

Then the system scales, and the arithmetic turns on it.

An agent generates approval requests at machine speed. A human reads them at
human speed. When the queue exceeds what a person can actually consider, the
behaviour does not degrade gracefully into "slower but careful". It degrades
into **approving without reading**, because the alternative is being the reason
nothing shipped.

The failure is invisible from inside the system. Every approval is recorded. The
audit trail shows a human decision on every action. The control appears to be
operating at 100%, and the thing being measured — that a human clicked — is not
the thing anyone cared about.

There is a second-order effect worth naming: an attacker who wants one approval
can *manufacture the volume that makes it likely*. Generate two hundred benign
requests, put the one that matters at position 173, and the control has been
defeated by arithmetic rather than by cleverness.

Approval is a control for irreversible actions. Used as a general-purpose gate
it becomes a click, and the risk register still counts it.

> **Where this lands on the reference architecture.**
>
> ```
> ingress -> orchestrator -> agent_runtime -> model
>                                |              |
>                          messaging        tools / mcp
>                                |              |
>                       knowledge / memory   egress
>            identity + policy wrap every call · observability records it
> ```

## 3 · The risk, realised

Approval quality against volume, and the position an attacker chooses.

In [ ]:
def review(queue, careful_capacity=25):
    """A reviewer considers the first `careful_capacity` properly. The rest are
    approved to keep the queue moving - which is what actually happens."""
    considered = queue[:careful_capacity]
    rubber_stamped = queue[careful_capacity:]
    caught = [r for r in considered if r["malicious"]]
    missed = [r for r in rubber_stamped if r["malicious"]]
    return {"considered": len(considered), "rubber_stamped": len(rubber_stamped),
            "caught": len(caught), "missed": len(missed)}

def build_queue(volume, malicious_at):
    return [{"id": i, "malicious": i == malicious_at} for i in range(volume)]

print(f"{'daily volume':>13}{'considered':>12}{'stamped':>9}{'caught':>8}{'missed':>8}")
for volume in (10, 25, 100, 500):
    q = build_queue(volume, malicious_at=volume - 1)     # attacker picks the position
    r = review(q)
    print(f"{volume:>13}{r['considered']:>12}{r['rubber_stamped']:>9}"
          f"{r['caught']:>8}{r['missed']:>8}")

print()
print("At every volume the audit trail shows a human approval on 100% of")
print("actions. The control reports full coverage in all four rows.")
print()
print("The attacker does not need to defeat the reviewer. They need to arrive")
print("at position 173 of 200, and generating positions 1 to 172 is free.")
r = review(build_queue(500, malicious_at=499))
assert r["missed"] == 1 and r["caught"] == 0

## What you just proved

Approval coverage reads 100% at every volume while the malicious request is caught only when the queue is small enough to be read — and an attacker choosing the position needs only to generate the requests in front of it.

## Your turn

Count how many approval requests one of your agents generates per day and ask the person approving them how many they read in full. The gap between those two numbers is the control's real coverage.

---

**Next → [A1.15 · Misaligned and deceptive behaviour](https://spbreed.github.io/cyber-commons/lessons/A1.15.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.14.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.14.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*